# PIPELINE PARALELO DE PROCESSAMENTO DE IMAGENS COM MULTIPROCESSING

INSTITUTO FEDERAL DE MINAS GERIAS Departamento de Engenharia e Computação

Alunos: Antonio Ambrosio, Euler Gomes e Vinicius Miguel


# 1. Preparação do ambiente

In [1]:
from IPython.display import display, HTML

display(HTML("<style>.container {widht: 100% !important;}</style>"))

## 1.1. Importação das bibliotecas

In [2]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import time
import shutil
import subprocess
import sys
import os
import kagglehub
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms

/home/euler/miniconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2. Exporta requirements

In [3]:
def exporta_requirements():
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                capture_output=True,
                                text=True,
                                check=True)
        with open('requirements.txt', 'w') as f:
            f.write(result.stdout)
        print('requirements.txt gerado com sucesso.')
    except subprocess.CalledProcessError as e:
        print('erro:', e)


exporta_requirements()

requirements.txt gerado com sucesso.


## 1.3. Checagem de GPU

In [4]:
if torch.cuda.is_available():
    print("__CUDNN VERSION:", torch.backends.cudnn.version())
    print("Device:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA não disponivel.")
    device = 'cpu'

print('Device:', device)

__CUDNN VERSION: 91501
Device: NVIDIA GeForce RTX 5070
Device: cuda


# 2. Carregar Dataset

## 2.1. Baixar dataset

In [5]:
baixar_dataset = True

if baixar_dataset:
    path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

    dest = Path("../parallel_loading/data/raw")
    dest.mkdir(parents=True, exist_ok=True)

    os.system(f"ln -s {path}/* {dest}")

ln: failed to create symbolic link '../parallel_loading/data/raw/HAM10000_images_part_1': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/HAM10000_images_part_2': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/HAM10000_metadata.csv': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/ham10000_images_part_1': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/ham10000_images_part_2': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/hmnist_28_28_L.csv': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/hmnist_28_28_RGB.csv': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/hmnist_8_8_L.csv': File exists
ln: failed to create symbolic link '../parallel_loading/data/raw/hmnist_8_8_RGB.csv': File exists


## 2.2. Juntar dataset

In [7]:
img_data1 = "data/raw/HAM10000_images_part_1"
img_data2 = "data/raw/HAM10000_images_part_2"
dest = "data/raw/images"

os.makedirs(dest, exist_ok=True)

for folder in [img_data1, img_data2]:
    for file in os.listdir(folder):
        shutil.copy(os.path.join(folder, file), dest)

## 2.3. Dividir dataset

In [9]:
df = pd.read_csv('data/raw/HAM10000_metadata.csv')
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
